In [1]:
import pandas as pd
import numpy as np
import os
from tqdm.notebook import tqdm

os.environ["KERAS_BACKEND"] = "tensorflow"

import implicit
from scipy.sparse import csr_matrix

2025-01-03 18:23:12.063121: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/jakub/anaconda3/envs/keras/lib/python3.10/site-packages/implicit/gpu/__init__.py:13: UserWarning: CUDA extension is built, but disabling GPU support because of 'Cuda Error: CUDA driver version is insufficient for CUDA runtime version (/home/conda/feedstock_root/build_artifacts/implicit_1724419599136/work/./implicit/gpu/utils.h:71)'
  warnings.warn(


# Prepare the data

In [2]:
game_folder = "data/game_data.csv"
time_folder = "data/time_data.csv"

game_data = pd.read_csv(game_folder)
time_data = pd.read_csv(time_folder)

In [3]:
# make the playtime in hours, remove games with less than 1 hour of playtime
time_data['playtime'] = time_data['playtime'] // 60
time_data = time_data[time_data['playtime'] != 0]
time_data.head()

,user_id,game_id,playtime
0,1,1,1346
1,1,2,10
2,1,3,8
3,1,4,10
4,1,5,6


In [4]:
game_time_sum = time_data.groupby('game_id')['user_id'].count()
game_time_sum = game_time_sum.reset_index()
game_time_sum.columns = ['game_id', 'users']
game_time_sum = game_time_sum.sort_values(by='users', ascending=False)
# get ids of games with at least 50 users, and filter the time_data
game_time_sum = game_time_sum[game_time_sum['users'] >= 50]
game_time_sum = game_time_sum['game_id'].values
time_data = time_data[time_data['game_id'].isin(game_time_sum)]
time_data.shape

(1123490, 3)

In [5]:
# get rid of game_ids that are called unknown_game in the game_data
game_data_ids = game_data[game_data['game_name'] != 'unknown game']["game_id"].values
# filter the time_data
time_data = time_data[time_data['game_id'].isin(game_data_ids)]
time_data.shape

(1079006, 3)

In [7]:
# map user_id and game_id to a unique integer
user_ids = time_data["user_id"].unique().tolist()
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
userencoded2user = {i: x for i, x in enumerate(user_ids)}
game_ids = time_data["game_id"].unique().tolist()
game2game_encoded = {x: i for i, x in enumerate(game_ids)}
game_encoded2game = {i: x for i, x in enumerate(game_ids)}
time_data["user_id"] = time_data["user_id"].map(user2user_encoded)
time_data["game_id"] = time_data["game_id"].map(game2game_encoded)

# get the number of users and games
num_users = len(user2user_encoded)
num_games = len(game_encoded2game)

# check the stats of the data
time_data["playtime"] = time_data["playtime"].values.astype(np.float32)
min_playtime = min(time_data["playtime"])
max_playtime = max(time_data["playtime"])

print(
    f"Number of users: {num_users}, Number of Games: {num_games}, Min playtime: {min_playtime}, Max playtime: {max_playtime}"
)


Number of users: 6560, Number of Games: 6113, Min playtime: 1.0, Max playtime: 75592.0


In [8]:
time_data = time_data.sample(frac=1, random_state=42)
x = time_data[["user_id", "game_id"]].values
y = time_data["playtime"].values

# 90/10 split
train_indices = int(0.9 * time_data.shape[0])
x_train, x_val, y_train, y_val = (
    x[:train_indices],
    x[train_indices:],
    y[:train_indices],
    y[train_indices:],
)

In [11]:
user_indices = time_data["user_id"].values
game_indices = time_data["game_id"].values
playtimes = time_data["playtime"].values

train_indices = int(0.9 * time_data.shape[0])
user_train, user_val = user_indices[:train_indices], user_indices[train_indices:]
game_train, game_val = game_indices[:train_indices], game_indices[train_indices:]
playtimes_train, playtimes_val = playtimes[:train_indices], playtimes[train_indices:]



# Train the model

In [14]:
test_games_per_user = {}
for i in range(len(user_val)):
    user = user_val[i]
    game = game_val[i]
    if user not in test_games_per_user:
        test_games_per_user[user] = []
    test_games_per_user[user].append(game)

total = 0
for user in test_games_per_user:
    total += len(test_games_per_user[user])
average = total / len(test_games_per_user)
print("Average number of games in the test dataset per user:", average)

Average number of games in the test dataset per user: 18.416282642089094


In [18]:
model = implicit.als.AlternatingLeastSquares(factors=256,
                                             regularization=0.1,
                                             iterations=30,
                                             calculate_training_loss=True)

# sparse_matrix = csr_matrix((playtimes, (user_indices, game_indices)))
sparse_matrix = csr_matrix((playtimes_train, (user_train, game_train)))

model.fit(sparse_matrix)

  0%|          | 0/30 [00:00<?, ?it/s]

In [19]:
recommend_tally = 0
for iter, user_id in tqdm(enumerate(test_games_per_user), total=len(test_games_per_user)):
    recommendations = model.recommend(user_id, sparse_matrix[user_id], N=20)
    recommended_games_in_test = set(recommendations[0]) & set(test_games_per_user.get(user_id))
    recommend_tally += len(recommended_games_in_test)

avg = recommend_tally / len(test_games_per_user)
print(recommend_tally)
print(avg)

  0%|          | 0/5859 [00:00<?, ?it/s]

13454
2.2962962962962963


# Generate game recommendations

In [14]:
# get random user_id
user_id = np.random.choice(user_indices)

recommendations = model.recommend(user_id, sparse_matrix[user_id])

# get user's most played games
user_data = time_data[time_data['user_id'] == user_id]
user_data = user_data.sort_values(by='playtime', ascending=False)
user_data["game_id"] = user_data["game_id"].map(game_encoded2game)
user_data = user_data.merge(game_data, on='game_id')

game_indices = [game_encoded2game.get(x) for x in recommendations[0]]
game_data[game_data["game_id"].isin(game_indices)]

print("Most played games:")
counter = 0
for index, row in user_data.iterrows():
    print(row['game_name'], row['playtime'])
    counter += 1
    if counter > 15:
        break

print("\nRecommended games:")
for game_id in game_indices:
    game_name = game_data[game_data["game_id"] == game_id]["game_name"].values[0]
    print(game_name)

Most played games:
Counter-Strike 2 4605.0
Call of Duty: Modern Warfare 2 (2009) - Multiplayer 661.0
Guild Wars 2 352.0
Path of Exile 293.0
Mount & Blade: Warband 221.0
PAYDAY 2 217.0
Apex Legends 189.0
New World: Aeternum 180.0
Phasmophobia 156.0
Warhammer: Vermintide 2 152.0
Beat Saber 151.0
Darkest Dungeon® 141.0
SMITE 140.0
Ultra Street Fighter IV 138.0
Baldur's Gate 3 136.0
Terraria 136.0

Recommended games:
EVE Online
The Elder Scrolls Online
The Witcher 2: Assassins of Kings Enhanced Edition
The Binding of Isaac
Lost Ark
Fallout 76
Hell Let Loose
Tom Clancy's Rainbow Six Siege
The Elder Scrolls V: Skyrim Special Edition
Hogwarts Legacy
